# Telecom Cellular Radio Network Anomaly Detection (Isolation Forest)
This notebook demonstrates unsupervised anomaly detection across 1,000 cellular base stations in Algerian wilayas, analyzing radio latency surges, packet degradation, user traffic crowding, and carrier availability breaches.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml.data.generate_network import generate_network_dataset
from ml.anomaly.train import train_anomaly_model
from ml.anomaly.evaluate import evaluate_anomaly_detection

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 1. Load Cell Network Telemetry Dataset

In [2]:
df_cells = generate_network_dataset(num_cells=1000, contamination=0.035, random_seed=42)
print(f"Total Cells: {len(df_cells)}")
print(f"Anomalies: {df_cells['is_anomaly'].sum()}")
df_cells.describe()

## 2. Telemetry Distribution & Outlier Visualization

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

scatter = ax[0].scatter(
    df_cells['latency_ms'], 
    df_cells['packet_loss_pct'], 
    c=df_cells['is_anomaly'], 
    cmap='coolwarm', 
    alpha=0.85, 
    edgecolors='k',
    linewidths=0.5
)
ax[0].set_title('Cellular Latency (ms) vs Packet Loss (%)')
ax[0].set_xlabel('Round Trip Latency (ms)')
ax[0].set_ylabel('Packet Drop Rate (%)')

ax[1].scatter(
    df_cells['users'], 
    df_cells['traffic_mbps'], 
    c=df_cells['is_anomaly'], 
    cmap='coolwarm', 
    alpha=0.85,
    edgecolors='k',
    linewidths=0.5
)
ax[1].set_title('Connected Users (UEs) vs Sector Throughput (Mbps)')
ax[1].set_xlabel('Connected UEs')
ax[1].set_ylabel('Throughput (Mbps)')

plt.tight_layout()
plt.show()

## 3. Train Unsupervised Isolation Forest & Benchmark

In [4]:
specs = train_anomaly_model()
print("Model Specifications:")
for k, v in specs.items():
    print(f"  {k}: {v}")